# Traverse the JSON files under `1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA1_first_328_batch`, build a matrix, and use each key in `segments.clauses` as a row under the matrix's `Clause` column. The remaining matrix columns are the 11 regions: `[California, Texas, Germany, Turkey, Egypt, Vietnam, Nigeria, India, Saudi, Bangladesh, Pakistan]`. If `segments.clauses.key.presence_flag` is 1, set the corresponding matrix row to 1.

Iterate through all JSON files under the directory.

Extract `segments[*].clauses`.

Use every clause key that appears as a row value in the matrix's `Clause` column.

If a clause appears in any segment and `presence_flag == "1"`, mark that row as 1.

Because the current JSON has no field for region-specific decisions, the 11 region columns must temporarily receive the same value: if a clause appears and has value 1, set all 11 columns to 1; otherwise set them to 0. This interpretation matches the structure of the provided files.

## Create submatrices

In [49]:
# {f_position}\{s_position}
f_position = "AA1_first_batch"
s_position = "AA1_first_328_batch" # AA4_forth_100_batch, AA6_sisth_74_batch  ## AA1_first_328_batch, AA4_forth_100_batch, AA6_sisth_74_batch

In [5]:
# out_openai_pp_txt_2024_10_mapping_matrix

In [50]:
import os
import json
import pandas as pd
from pathlib import Path

In [ ]:
# ========= Configuration =========
# INPUT_DIR = r"1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\test"
# OUTPUT_DIR = os.path.join(INPUT_DIR, "matrix_output")
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# REGIONS = [
#     "California", "Texas", "Germany", "Turkey", "Egypt",
#     "Vietnam", "Nigeria", "India", "Saudi", "Bangladesh", "Pakistan"
# ]

REGION_COLUMN_ALIASES = {
    "California": ["usa_version_code"],
    "Texas": ["usa_version_code"],
    "Germany": ["germany_version_code"],
    "Turkey": ["turkey_version_code"],
    "Egypt": ["egypt_version_code"],
    "Vietnam": ["vietnam_version_code"],
    "Nigeria": ["nigeria_version_code"],
    "India": ["india_version_code"],
    "Saudi": ["saudi_version_code"],
    "Bangladesh": ["bangladesh_version_code"],
    "Pakistan": ["pakistan_version_code"],
}

In [52]:
ROOT_DIR = Path(r"dataset4geodiff/apk_versions_long.csv")

# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA3_third_100_batch
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA4_forth_100_batch
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA5_fifth_100_batch
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA6_sisth_74_batch
RAW_ROOT = Path(rf"1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\{f_position}\{s_position}")
# OUT_ROOT = Path(r"1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\two_matrix")
OUT_ROOT = Path(rf"1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\{f_position}\{s_position}")

TEMPLATE_XLSX = Path(r"dataset4geodiff\can_detect_11regions_regulations_pp.xlsx")

In [53]:
def load_matrix_template_axes(template_xlsx_path: Path):
    tpl = pd.read_excel(template_xlsx_path, dtype=str)
    if tpl.empty:
        raise ValueError(f"Template xlsx is empty: {template_xlsx_path}")

    id_col = "Clause" if "Clause" in tpl.columns else tpl.columns[0]
    print(f"ID Column: {id_col}")
    clause_axis = [
        str(x).strip()
        for x in tpl[id_col].tolist()
        if str(x).strip() and str(x).strip().lower() != "nan"
    ]
    region_axis = [str(c).strip() for c in tpl.columns if c != id_col]

    if not clause_axis:
        raise ValueError(f"No clause rows found in template: {template_xlsx_path}")
    if not region_axis:
        raise ValueError(f"No region columns found in template: {template_xlsx_path}")
    print(f"Clause Axis: {clause_axis}\n", "len(clause_axis):", len(clause_axis))
    print(f"Region Axis: {region_axis}\n", "len(region_axis):", len(region_axis))
    return clause_axis, region_axis


TEMPLATE_CLAUSE_AXIS, TEMPLATE_REGION_AXIS = load_matrix_template_axes(TEMPLATE_XLSX)
REGIONS = TEMPLATE_REGION_AXIS
print(REGIONS)

ID Column: Clause
Clause Axis: ['P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'CR1', 'CR2', 'CR3', 'CR4', 'CR6', 'R1', 'R2', 'R4', 'R5', 'R6', 'R7', 'R8', 'R9', 'R10', 'R11', 'R13', 'R14', 'R15', 'R16', 'R17', 'R19', 'R20', 'E1', 'E24', 'E25', 'E26', 'E27', 'E28', 'O1', 'O2', 'O3']
 len(clause_axis): 51
Region Axis: ['California', 'Texas', 'Germany', 'Turkey', 'Egypt', 'Vietnam', 'Nigeria', 'India', 'Saudi', 'Bangladesh', 'Pakistan']
 len(region_axis): 11
['California', 'Texas', 'Germany', 'Turkey', 'Egypt', 'Vietnam', 'Nigeria', 'India', 'Saudi', 'Bangladesh', 'Pakistan']


In [ ]:
def process_version(apk_version_df, apkname, baseline_version):
    apk_rows = apk_version_df[apk_version_df['apkname'] == apkname]
    if apk_rows.empty:
        print(f"[WARN] {apkname} not found in version mapping file")
        return None
    versions = apk_rows["version"].values
    for v in versions:
        if str(v).strip()==baseline_version:
            continue
        target_version = str(v).strip()
    if not target_version:
        print(f"[WARN] {apkname} has empty version in mapping file")
        return None

    return target_version

def find_one_file(folder: Path, pattern: str):
    matches = sorted(folder.glob(pattern))
    return matches[0] if matches else None


def find_two_file(folder: Path, pattern: str):
    matches = sorted(folder.glob(pattern))
    return matches if len(matches) > 1 else None


def _extract_mapping_items(obj):
    if isinstance(obj, dict):
        all_kept = obj.get("all_kept_mappings", [])
        if isinstance(all_kept, list):
            return all_kept
        return []
    if isinstance(obj, list):
        items = []
        for x in obj:
            if isinstance(x, dict):
                if "all_kept_mappings" in x:
                    v = x.get("all_kept_mappings", [])
                    if isinstance(v, list):
                        items.extend(v)
                else:
                    items.append(x)
        return items
    return []

def load_clause_ids_1(segments_mapping_path: Path):
    with open(segments_mapping_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    items = []
    if isinstance(data, dict):
        pmr = data.get("segments", {})
        items = _extract_mapping_items(pmr)
    elif isinstance(data, list):
        for entry in data:
            if not isinstance(entry, dict):
                continue
            if "path_mapping_results" in entry:
                items.extend(_extract_mapping_items(entry.get("path_mapping_results")))
            else:
                items.extend(_extract_mapping_items(entry))

    clause_ids = []
    for item in items:
        if not isinstance(item, dict):
            continue
        cid = str(item.get("ours_id", "")).strip()
        if cid:
            clause_ids.append(cid)
    return sorted(set(clause_ids))

def load_clause_ids(segments_mapping_path: Path):
    with open(segments_mapping_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        detected_clauses = set()
        for segment in data.get("segments_mapping_results", []):
            clauses_dict = segment.get("clauses", {})
            detected_clauses.update(clauses_dict.keys())
            # print(f"Extracted Clause IDs: {detected_clauses}")
            return sorted(detected_clauses)

def find_target_version(all_info_df: pd.DataFrame, app_id: str, baseline: str) -> str:
    # Find a version different from baseline among all APK version records.
    apk = all_info_df[all_info_df["apkname"] == app_id]
    if apk.empty:
        return ""
    vals = apk["version"].astype(str).str.strip().unique().tolist()
    uniq = []
    for v in vals:
        if v not in uniq:
            if v != baseline:
                uniq.append(v)
    print(app_id,uniq)
    return uniq[0] if uniq else ""

def load_versions(ROOT_DIR: Path, country_csv_path: Path, apkname: str):
    all_apks_info = pd.read_csv(ROOT_DIR, dtype=str)
    # 1) Read country_versions.csv.
    try:
        df = pd.read_csv(country_csv_path, dtype=str).fillna("")
    except Exception as e:
        print(f"[SKIP] {country_csv_path}: failed to read CSV -> {e}")
        return "", ""

    if df.empty:
        print(f"[SKIP] {country_csv_path}: CSV is empty")
        return "", ""

    row = df.iloc[0]
    baseline_version = str(row.get("usa_version_code", "")).strip()

    target_version = find_target_version(all_apks_info, apkname, baseline_version)

    return baseline_version, target_version

def find_version1(file_name: str):
    suffix = ".json"

    base_name = file_name.replace(suffix, "")
    if "_" in base_name:
        apkname, version = base_name.rsplit("_", 1)
        return version.strip()
    return ""

def find_version(file_name: str):
    file_name = Path(file_name)
    suffix = file_name.stem
    return suffix

def get_region_candidate_columns(region: str):
    aliases = REGION_COLUMN_ALIASES.get(region, [])
    default_col = f"{region.lower()}_version_code"
    # print(f"Region '{region}' candidate columns: {aliases + [default_col]}")
    return list(dict.fromkeys(aliases + [default_col]))


def regions_for_version(country_df: pd.DataFrame, version: str):
    target_regions = []
    v = str(version).strip()
    if not v:
        return target_regions

    for region in REGIONS:
        candidate_cols = get_region_candidate_columns(region)
        matched = False
        for col in candidate_cols:
            if col not in country_df.columns:
                continue
            series = country_df[col].astype(str).str.strip()
            if (series == v).any():
                matched = True
                break
        if matched:
            target_regions.append(region)
    return target_regions


def build_matrix_1(detected_clause_ids, active_regions):
    matrix = pd.DataFrame(0, index=TEMPLATE_CLAUSE_AXIS, columns=TEMPLATE_REGION_AXIS, dtype=int)
    matrix.index.name = "Clause" # "clause_id"

    clause_set = set(detected_clause_ids)
    region_set = set(active_regions)

    for cid in TEMPLATE_CLAUSE_AXIS:
        if cid not in clause_set:
            continue
        for region in TEMPLATE_REGION_AXIS:
            if region in region_set:
                matrix.at[cid, region] = 1
    return matrix


def build_matrix(detected_clause_ids, active_regions):
    # 1. Deduplicate and sort column names to keep the matrix structure stable.
    # If the original input order should be preserved, use list(dict.fromkeys(active_regions)) instead.
    sorted_active_regions = sorted(list(set(active_regions)))
    
    # 2. Initialize the matrix columns using the provided regions.
    # The index still uses the predefined TEMPLATE_CLAUSE_AXIS.
    matrix = pd.DataFrame(
        0, 
        index=TEMPLATE_CLAUSE_AXIS, 
        columns=sorted_active_regions, 
        dtype=int
    )
    matrix.index.name = "Clause"

    # Convert detected clauses to a set for faster lookup.
    clause_set = set(detected_clause_ids)
    
    # 3. Fill the matrix.
    # Since the columns are active_regions, iterate through detected_clause_ids.
    for cid in clause_set:
        if cid in matrix.index:
            # Set all active_regions columns in the row to 1.
            matrix.loc[cid, :] = 1
            
    return matrix

# Example call.
# matrix_df = build_matrix(clause_ids, active_regions)

def process_one_apk_folder(apk_folder: Path):
    apkname = apk_folder.name
    print(f"[INFO] Processing APK folder: apkname: {apkname}, {apk_folder}. ")

    if not os.path.isdir(apk_folder):
        # print(f"{apk_name} is not a folder; skip it")
        pass

    process_dir = os.path.join(apk_folder, 'p')
    print(f"[INFO] process_dir: {process_dir}")

    # single_json_records = []

    # 2. Check whether the process folder exists.
    if os.path.exists(process_dir) and os.path.isdir(process_dir):
    

        # 4. Find files ending with .json.
        segments_mapping_path_list = find_two_file(Path(process_dir), "*.json")
        # print(f"[INFO]: found segments_mapping_path_list: {segments_mapping_path_list}")

        if segments_mapping_path_list is None or len(segments_mapping_path_list) <= 1:
            # single_json_records.append({
            #     "process_dir": str(process_dir),
            #     "json_file": str(segments_mapping_path_list[0]),
            #     "reason": "only_one_json_found"
            # })
            print(f"[SKIP] Only one JSON found in {process_dir}: {segments_mapping_path_list}")
        else:
            for segments_mapping_path in segments_mapping_path_list:
                print(f"[INFO] Processing segments mapping file: {segments_mapping_path}")
                country_csv_path = find_one_file(Path(process_dir), "*.csv")

                if segments_mapping_path is None:
                    pass
                if country_csv_path is None:
                    pass

                # Load the JSON file and extract clause IDs.
                clause_ids = load_clause_ids(segments_mapping_path)
                print(f"[INFO] {apkname}: detected clause ids: {clause_ids}")

                if clause_ids is None:
                    print(f"[SKIP] {apkname}: failed to load clause ids from {segments_mapping_path}")
                    continue


                # print(f"[INFO] segments_mapping_path.name:{segments_mapping_path.name}")
                version = find_version(segments_mapping_path.name)
                print(f"[INFO] {apkname}: version={version}")


                country_df = pd.read_csv(country_csv_path, dtype=str)
                country_df.drop(columns=['package_name'], inplace=True)
                print(f"{country_df.head()}")


                active_regions = regions_for_version(country_df, version)
                print(apkname, active_regions)

                matrix_df = build_matrix(clause_ids, active_regions)
                print(matrix_df.shape)

                # out_dir = os.path.join(apk_folder, 'two_matrix')
                out_dir = apk_folder.parent / "two_matrix"
                out_dir = Path(out_dir)
                out_dir.mkdir(parents=True, exist_ok=True)
                
                out_path = os.path.join(out_dir, f"{apkname}_{version}.csv") #f"{apkname}_{version}.csv"
                print(f"[INFO] out_path: {out_path}")

                matrix_df.to_csv(out_path, encoding="utf-8-sig")


                print(
                    f"[OK] {apkname}: clauses_detected={len(clause_ids)}, "
                    f"template_rows={len(TEMPLATE_CLAUSE_AXIS)}, template_cols={len(TEMPLATE_REGION_AXIS)}, "
                    f"version={version}, active_regions={active_regions}, saved={out_path}"
                )

In [ ]:
# # # Batch-iterate through the first-level APK folders.
for apk_folder in sorted([p for p in OUT_ROOT.iterdir() if p.is_dir()]):
    apk_name = apk_folder.name
    
    # skip folders that are not APK package names
    if "." not in apk_name:
        # print(f"[SKIP] Not an APK folder: {apk_folder}")
        continue

    # if apk_name in["com.EternalStudio.SurvivorZ", "com.halfbrick.fruitninjax", "com.hecorat.screenrecorder.free", "com.herocraft.game.free.stww2_sandbox", "com.hiroba.helix"]: 
    # if apk_name == "com.halfbrick.fruitninjax":
    process_one_apk_folder(apk_folder)
    # print(f"Processing APK folder: {apk_name}")
        
# com.benoitletondor.pixelminimalwatchface
# process_one_apk_folder(Path(r"1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\test\com.brainium.solitairefree")) # com.bandagames.mpuzzle.gp ae.brandsforless.android  air.bg.lan.Monopoli  air.com.bigwigmedia.hotdogbush

[INFO] Processing APK folder: apkname: com.lydia, 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA6_sisth_74_batch\com.lydia. 
[INFO] process_dir: 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA6_sisth_74_batch\com.lydia\p
[INFO] Processing segments mapping file: 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA6_sisth_74_batch\com.lydia\p\55576.json
[INFO] com.lydia: detected clause ids: ['O1', 'O2', 'P11', 'P4', 'P8']
[INFO] com.lydia: version=55576
  germany_version_code vietnam_version_code nigeria_version_code  \
0                55589                55589                55589   

  pakistan_version_code india_version_code usa_version_code  \
0                 55589              55576            55576   

  turkey_version_code egypt_version_code bangladesh_version_code  \
0               55576              55589                   55589   

  saudi_version_code  
0              55589  
com.

## Create the remaining submatrices when there are three or more versions
Input: `RAW_ROOT = Path(r"1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\")`.

`input_pth = Path(r"1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\{apk_name}\two_matrix")` contains the two already-created submatrices and their version numbers.

`input_pth2 = Path(r"1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\{apk_name}\process")` contains the app's version numbers in different countries. Use `country_csv_path = find_one_file(Path(process_dir), "*.csv")` to obtain all versions for the app.

Exclude the version numbers of the two existing submatrices, then create a submatrix for the countries corresponding to each remaining version.

Save to `out_pth = Path(r"1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\{apk_name}\two_matrix\{apk_name}_{version}.csv")`.

In [ ]:
# from pathlib import Path

# import pandas as pd


# RAW_ROOT = Path(r"1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch")

# REGION_COLUMN_ALIASES = {
#     "California": "usa_version_code",
#     "Texas": "usa_version_code",
#     "Germany": "germany_version_code",
#     "Turkey": "turkey_version_code",
#     "Egypt": "egypt_version_code",
#     "Vietnam": "vietnam_version_code",
#     "Nigeria": "nigeria_version_code",
#     "India": "india_version_code",
#     "Saudi": "saudi_version_code",
#     "Bangladesh": "bangladesh_version_code",
#     "Pakistan": "pakistan_version_code",
# }

In [ ]:

# def load_matrix_template_axes(template_xlsx_path: Path):
#     tpl = pd.read_excel(template_xlsx_path, dtype=str)
#     if tpl.empty:
#         raise ValueError(f"Template xlsx is empty: {template_xlsx_path}")

#     id_col = "Clause" if "Clause" in tpl.columns else tpl.columns[0]
#     print(f"ID Column: {id_col}")
#     clause_axis = [
#         str(x).strip()
#         for x in tpl[id_col].tolist()
#         if str(x).strip() and str(x).strip().lower() != "nan"
#     ]
#     region_axis = [str(c).strip() for c in tpl.columns if c != id_col]

#     if not clause_axis:
#         raise ValueError(f"No clause rows found in template: {template_xlsx_path}")
#     if not region_axis:
#         raise ValueError(f"No region columns found in template: {template_xlsx_path}")
#     print(f"Clause Axis: {clause_axis}\n", "len(clause_axis):", len(clause_axis))
#     print(f"Region Axis: {region_axis}\n", "len(region_axis):", len(region_axis))
#     return clause_axis, region_axis


# TEMPLATE_CLAUSE_AXIS, TEMPLATE_REGION_AXIS = load_matrix_template_axes(TEMPLATE_XLSX)
# REGIONS = TEMPLATE_REGION_AXIS
# print(REGIONS)

ID Column: Clause
Clause Axis: ['P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'CR1', 'CR2', 'CR3', 'CR4', 'CR6', 'R1', 'R2', 'R4', 'R5', 'R6', 'R7', 'R8', 'R9', 'R10', 'R11', 'R13', 'R14', 'R15', 'R16', 'R17', 'R19', 'R20', 'E1', 'E24', 'E25', 'E26', 'E27', 'E28', 'O1', 'O2', 'O3']
 len(clause_axis): 51
Region Axis: ['California', 'Texas', 'Germany', 'Turkey', 'Egypt', 'Vietnam', 'Nigeria', 'India', 'Saudi', 'Bangladesh', 'Pakistan']
 len(region_axis): 11
['California', 'Texas', 'Germany', 'Turkey', 'Egypt', 'Vietnam', 'Nigeria', 'India', 'Saudi', 'Bangladesh', 'Pakistan']


In [ ]:
# def find_one_file(folder: Path, pattern: str) -> Path:
#     files = list(folder.glob(pattern))

#     # if len(files) != 1:
#     #     raise ValueError(
#     #         f"Expected one {pattern} in {folder}, found {len(files)}"
#     #     )

#     return files[0]


# # def load_matrix_template_axes(template_xlsx_path: Path):
# #     tpl = pd.read_excel(template_xlsx_path, dtype=str)

# #     id_col = "Clause" if "Clause" in tpl.columns else tpl.columns[0]

# #     clause_axis = tpl[id_col].dropna().astype(str).str.strip().tolist()
# #     region_axis = [column for column in tpl.columns if column != id_col]

# #     return clause_axis, region_axis


# def process_one_apk_folder(apk_folder: Path):
#     apk_name = apk_folder.name
#     process_dir = apk_folder / "process"
#     two_matrix_dir = apk_folder / "two_matrix"

#     if not process_dir.exists() or not two_matrix_dir.exists():
#         return

#     # Read the App's version numbers in different regions.
#     country_csv_path = find_one_file(process_dir, "*.csv")
#     version_row = pd.read_csv(country_csv_path, dtype=str).iloc[0]

#     # version -> regions
#     version_regions = {}

#     for region in REGIONS:
#         version_column = REGION_COLUMN_ALIASES[region]

#         if version_column not in version_row:
#             continue

#         version = version_row[version_column]

#         if pd.isna(version) or not str(version).strip():
#             continue

#         version = str(version).strip()
#         version_regions.setdefault(version, []).append(region)

#     # Do not process apps with fewer than three versions.
#     if len(version_regions) < 3:
#         return

#     # Read the already-created version numbers from the existing submatrix filenames.
#     existing_versions = {
#         file_path.stem.removeprefix(f"{apk_name}_")
#         for file_path in two_matrix_dir.glob(f"{apk_name}_*.csv")
#     }

#     # Create an empty submatrix for each remaining version.
#     for version, regions in version_regions.items():
#         if version in existing_versions:
#             continue

#         out_path = two_matrix_dir / f"{apk_name}_{version}.csv"

#         submatrix = pd.DataFrame(
#             "",
#             index=TEMPLATE_CLAUSE_AXIS,
#             columns=regions,
#         )
#         submatrix.index.name = "Clause"

#         submatrix.to_csv(out_path, encoding="utf-8-sig")

#         print(
#             f"[CREATED] {out_path.name}: "
#             f"{', '.join(regions)}"
#         )

In [ ]:
# # # # Batch iterate through the top-level APK folders.
# for apk_folder in sorted([p for p in RAW_ROOT.iterdir() if p.is_dir()]):
#     apk_name = apk_folder.name
    
#     # skip folders that are not APK package names
#     if "." not in apk_name:
#         # print(f"[SKIP] Not an APK folder: {apk_folder}")
#         continue

#     if apk_name == "com.EternalStudio.SurvivorZ": 
#         # print(f"Processing APK folder: {apk_folder}")
#         process_one_apk_folder(apk_folder)

[CREATED] com.EternalStudio.SurvivorZ_55.csv: Nigeria, Saudi, Pakistan


Path: `1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\two_matrix\com.brainium.solitairefree.csv`

## Merge submatrices

Read the folder `1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\two_matrix`. It contains CSV files whose names include the apkname and version. Group files with the same apkname to build one final matrix.

Load each CSV file separately and combine them into the final document.

In [ ]:
from pathlib import Path
import pandas as pd
import re

def get_apkname_from_filename(fp: Path) -> str:
    """
    Remove the version from the filename to obtain the apkname.

    Supported formats:
    com.xxx.app_12345.csv        -> com.xxx.app
    com.xxx.app_1.2.3.csv        -> com.xxx.app
    com.xxx.app_v1.2.3.csv       -> com.xxx.app
    """

    stem = fp.stem

    # Remove the version after the final underscore.
    # Example: com.xxx.app_12345 -> com.xxx.app
    # Example: com.xxx.app_1.2.3 -> com.xxx.app
    # Example: com.xxx.app_v1.2.3 -> com.xxx.app
    apkname = re.sub(r"_(v?\d+[\d.]*)$", "", stem)
    print(f"Extracted apkname: {apkname} ") #from filename: {fp.name}

    return apkname

def load_matrix(fp: Path) -> pd.DataFrame:
    """
    Read a matrix CSV.
    The first column is assumed to be the Clause/index column.
    """

    df = pd.read_csv(fp, index_col=0)

    # Remove possible empty rows and columns.
    df = df.dropna(how="all")
    df = df.dropna(axis=1, how="all")

    # Convert to numeric where possible; replace non-convertible values with 0.
    df = df.apply(pd.to_numeric, errors="coerce").fillna(0)

    # Normalize values to 0/1.
    df = (df > 0).astype(int)

    return df

In [ ]:
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\two_matrix
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA3_third_100_batch\two_matrix
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA4_forth_100_batch\two_matrix
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA5_fifth_100_batch\two_matrix
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA6_sisth_74_batch\two_matrix
root_dir = Path(
    rf"1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\{f_position}\{s_position}\two_matrix"
)

# 1. Group by apkname.
apk_groups = {}
for fp in root_dir.glob("*.csv"):
    apkname = get_apkname_from_filename(fp)
    apk_groups.setdefault(apkname, []).append(fp)

print(f"[INFO] Found {len(apk_groups)} apk groups.")

Extracted apkname: com.lydia 
Extracted apkname: com.lydia 
Extracted apkname: com.magdalm.freewifipassword 
Extracted apkname: com.magdalm.freewifipassword 
Extracted apkname: com.map.photostamp 
Extracted apkname: com.map.photostamp 
Extracted apkname: com.mars.avgchapters 
Extracted apkname: com.mars.avgchapters 
Extracted apkname: com.masmovil.masmovil 
Extracted apkname: com.masmovil.masmovil 
Extracted apkname: com.mason.wooplus 
Extracted apkname: com.mason.wooplus 
Extracted apkname: com.mate.vpn 
Extracted apkname: com.mate.vpn 
Extracted apkname: com.maxis.mymaxis 
Extracted apkname: com.maxis.mymaxis 
Extracted apkname: com.mcdo.mcdonalds 
Extracted apkname: com.mcdo.mcdonalds 
Extracted apkname: com.memeandsticker.personal 
Extracted apkname: com.memeandsticker.personal 
Extracted apkname: com.mg.bowman.attack3d.archer 
Extracted apkname: com.mg.bowman.attack3d.archer 
Extracted apkname: com.mg.wild.gunfighter.west.sniper 
Extracted apkname: com.mg.wild.gunfighter.west.snip

In [ ]:
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\merged_two_matrix
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA3_third_100_batch\merged_two_matrix
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA4_forth_100_batch\merged_two_matrix
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA5_fifth_100_batch\merged_two_matrix
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA6_sisth_74_batch\merged_two_matrix
root_apk_dir = Path(rf"1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\{f_position}\{s_position}\merged_two_matrix")
root_apk_dir.mkdir(parents=True, exist_ok=True)

# 2. Merge the matrices for multiple versions under each apkname.
for apkname, files in apk_groups.items():
    print(f"\n[INFO] Processing apk: {apkname}")
    print(f"[INFO] Files: {[f.name for f in files]}")

    all_dfs = []

    for csv_fp in files:
        try:
            df = pd.read_csv(csv_fp, index_col=0)
            all_dfs.append(df)
        except Exception as e:
            print(f"Failed to read {csv_fp}: {e}")

    if all_dfs:
        # merged_df = pd.concat(all_dfs, ignore_index=True, sort=False)
        merged_df = pd.concat(all_dfs, axis=1)

        out_fp = root_apk_dir / f"{apkname}.csv"
        print(f"Saving merged dataframe to: {out_fp}")
        merged_df.to_csv(out_fp, index=True, encoding="utf-8-sig")

        # print(merged_df.head(), merged_df.shape)
    else:
        merged_df = pd.DataFrame()
        print("No csv files found.")


[INFO] Processing apk: com.lydia
[INFO] Files: ['com.lydia_55576.csv', 'com.lydia_55589.csv']
Saving merged dataframe to: 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA6_sisth_74_batch\merged_two_matrix\com.lydia.csv

[INFO] Processing apk: com.magdalm.freewifipassword
[INFO] Files: ['com.magdalm.freewifipassword_1401.csv', 'com.magdalm.freewifipassword_1403.csv']
Saving merged dataframe to: 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA6_sisth_74_batch\merged_two_matrix\com.magdalm.freewifipassword.csv

[INFO] Processing apk: com.map.photostamp
[INFO] Files: ['com.map.photostamp_100.csv', 'com.map.photostamp_106.csv']
Saving merged dataframe to: 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA6_sisth_74_batch\merged_two_matrix\com.map.photostamp.csv

[INFO] Processing apk: com.mars.avgchapters
[INFO] Files: ['com.mars.avgchapters_6620.csv', 'com.mars.avgchapters_6630.csv']
Saving merged d

check root_apk_dir = Path(r"1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\merged_two_matrix") load every csv and print the dataframe size

In [41]:
from pathlib import Path
import pandas as pd

# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\merged_two_matrix
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA3_third_100_batch\merged_two_matrix
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA4_forth_100_batch\merged_two_matrix
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA5_fifth_100_batch\merged_two_matrix
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA6_sisth_74_batch\merged_two_matrix
root_apk_dir = Path(
    rf"1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10"
    rf"\{f_position}\{s_position}\merged_two_matrix"
)

for csv_fp in root_apk_dir.glob("*.csv"):
    try:
        df = pd.read_csv(csv_fp, index_col=0)

        print(f"[OK] {csv_fp.name}")
        print(f"     shape = {df.shape}")
        print(f"     rows  = {df.shape[0]}")
        print(f"     cols  = {df.shape[1]}")

    except Exception as e:
        print(f"[ERROR] {csv_fp.name}: {e}")

[OK] com.lydia.csv
     shape = (51, 11)
     rows  = 51
     cols  = 11
[OK] com.magdalm.freewifipassword.csv
     shape = (51, 11)
     rows  = 51
     cols  = 11
[OK] com.map.photostamp.csv
     shape = (51, 11)
     rows  = 51
     cols  = 11
[OK] com.mars.avgchapters.csv
     shape = (51, 11)
     rows  = 51
     cols  = 11
[OK] com.masmovil.masmovil.csv
     shape = (51, 11)
     rows  = 51
     cols  = 11
[OK] com.mason.wooplus.csv
     shape = (51, 11)
     rows  = 51
     cols  = 11
[OK] com.mate.vpn.csv
     shape = (51, 11)
     rows  = 51
     cols  = 11
[OK] com.maxis.mymaxis.csv
     shape = (51, 11)
     rows  = 51
     cols  = 11
[OK] com.mcdo.mcdonalds.csv
     shape = (51, 11)
     rows  = 51
     cols  = 11
[OK] com.memeandsticker.personal.csv
     shape = (51, 11)
     rows  = 51
     cols  = 11
[OK] com.mg.bowman.attack3d.archer.csv
     shape = (51, 11)
     rows  = 51
     cols  = 11
[OK] com.mg.wild.gunfighter.west.sniper.csv
     shape = (51, 11)
     rows  = 5

move size of csv not (51,11) to merged_matrix_not

In [ ]:
# from pathlib import Path
# import pandas as pd
# import shutil

# # 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\merged_two_matrix
# # 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA3_third_100_batch\merged_two_matrix
# # 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA4_forth_100_batch\merged_two_matrix
# # 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA5_fifth_100_batch\merged_two_matrix
# # 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA6_sisth_74_batch\merged_two_matrix
# root_apk_dir = Path(
#     r"1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10"
#     r"\AA1_first_batch\AA3_third_100_batch\merged_two_matrix"
# )

# # 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\merged_two_matrix
# # 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA3_third_100_batch\merged_two_matrix
# # 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA4_forth_100_batch\merged_two_matrix
# # 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA5_fifth_100_batch\merged_two_matrix
# # 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA6_sisth_74_batch\merged_two_matrix
# not_dir = Path(
#     r"1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10"
#     r"\AA1_first_batch\AA3_third_100_batch\merged_matrix_not"
# )
# not_dir.mkdir(parents=True, exist_ok=True)

# expected_shape = (51, 11)

# for csv_fp in root_apk_dir.glob("*.csv"):
#     try:
#         df = pd.read_csv(csv_fp, index_col=0)

#         status = "OK" if df.shape == expected_shape else "CHECK"

#         print(f"[{status}] {csv_fp.name}: shape = {df.shape}")

#         if status == "CHECK":
#             dst_fp = not_dir / csv_fp.name

#             # Avoid overwriting an existing file with the same name in the target directory.
#             if dst_fp.exists():
#                 dst_fp = not_dir / f"{csv_fp.stem}_duplicate{csv_fp.suffix}"

#             shutil.move(str(csv_fp), str(dst_fp))
#             print(f"     moved to: {dst_fp}")
            

#     except Exception as e:
#         print(f"[ERROR] {csv_fp.name}: {e}")

[OK] com.brainium.solitairefree.csv: shape = (51, 11)
[OK] com.braintest.happy.ending.csv: shape = (51, 11)
[OK] com.brave.merge.csv: shape = (51, 11)
[CHECK] com.breakingnewsbrief.app.csv: shape = (51, 10)
     moved to: 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\merged_matrix_not\com.breakingnewsbrief.app.csv
[OK] com.bubbleshooter.popbubbles.shootbubblesgame.csv: shape = (51, 11)
[CHECK] com.bydeluxe.d3.android.program.starz.csv: shape = (51, 3)
     moved to: 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\merged_matrix_not\com.bydeluxe.d3.android.program.starz.csv
[OK] com.CalendarMonthlyStyle.csv: shape = (51, 11)
[OK] com.CharityRun.game.csv: shape = (51, 11)
[OK] com.DefaultCompany.BloodBox.csv: shape = (51, 11)
[OK] com.Dekovir.PixWordsScenes.csv: shape = (51, 11)
[CHECK] com.EternalStudio.SurvivorZ.csv: shape = (51, 8)
     moved to: 1122apk\1122apk_privacy_policy_url\out_

statistic the final apkname in path: "1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\merged_two_matrix"

The following script will read all `.csv` files within the `merged_two_matrix` directory, use the filename stems as the `apkname`, and then save the results as a consolidated CSV file.

In [54]:
from pathlib import Path
import pandas as pd

# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\merged_two_matrix
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA3_third_100_batch\merged_two_matrix
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA4_forth_100_batch\merged_two_matrix
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA5_fifth_100_batch\merged_two_matrix
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA6_sisth_74_batch\merged_two_matrix
root_apk_dir = Path(
    rf"1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10"
    rf"\{f_position}\{s_position}\merged_two_matrix"
)

# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\merged_two_matrix\apkname_list.csv
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA3_third_100_batch\merged_two_matrix\apkname_list.csv
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA4_forth_100_batch\merged_two_matrix\apkname_list.csv
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA5_fifth_100_batch\merged_two_matrix\apkname_list.csv
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA6_sisth_74_batch\merged_two_matrix\apkname_list.csv
out_fp = Path(
    rf"1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10"
    rf"\{f_position}\{s_position}\merged_two_matrix\apkname_list.csv"
)

apkname_list = []

for csv_fp in root_apk_dir.glob("*.csv"):
    apkname = csv_fp.stem
    apkname_list.append(apkname)

df_out = pd.DataFrame({
    "apkname": sorted(apkname_list)
})

df_out.to_csv(out_fp, index=False, encoding="utf-8-sig")

print(f"[OK] Found {len(df_out)} apkname")
print(f"[OK] Saved to: {out_fp}")

[OK] Found 46 apkname
[OK] Saved to: 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA1_first_328_batch\merged_two_matrix\apkname_list.csv


<!--
Path(r"1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\test\com.brainium.solitairefree\two_matrix")

Read the folder `1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\test`. Each folder is named after an apkname and contains a `two_matrix` folder and several JSON files. Open the `two_matrix` folder, which contains several CSV files.

Load the CSV files separately and combine them into the final document. -->

In [ ]:
# from pathlib import Path
# import pandas as pd

# root_dir = Path(
#     r"1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\two_matrix"
# )

# root_apk_dir = ""

# all_dfs = []

# for apk_dir in root_dir.iterdir():
#     if not apk_dir.is_dir():
#         continue

#     root_apk_dir = apk_dir

#     apk_name = apk_dir.name
#     two_matrix_dir = apk_dir / "two_matrix"

#     if not two_matrix_dir.is_dir():
#         print(f"Skip {apk_name}: no two_matrix folder")
#         continue

#     csv_files = list(two_matrix_dir.glob("*.csv"))
#     if not csv_files:
#         print(f"Skip {apk_name}: no csv files")
#         continue

#     for csv_fp in csv_files:
#         try:
#             df = pd.read_csv(csv_fp, index_col=0)
#             all_dfs.append(df)
#         except Exception as e:
#             print(f"Failed to read {csv_fp}: {e}")

# if all_dfs:
#     # merged_df = pd.concat(all_dfs, ignore_index=True, sort=False)
#     merged_df = pd.concat(all_dfs, axis=1)

#     out_fp = root_apk_dir / "merged_two_matrix.csv"
#     print(f"Saving merged dataframe to: {out_fp}")
#     merged_df.to_csv(out_fp, index=True, encoding="utf-8-sig")

#     # print(merged_df.head(), merged_df.shape)
# else:
#     merged_df = pd.DataFrame()
#     print("No csv files found.")

Saving merged dataframe to: 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch\test\com.brainium.solitairefree\merged_two_matrix.csv
        California  India  Texas  Vietnam  Bangladesh  Egypt  Germany  \
Clause                                                                  
P1               0      0      0        0           1      1        1   
P2               1      1      1        1           1      1        1   
P3               1      1      1        1           1      1        1   
P4               1      1      1        1           1      1        1   
P5               1      1      1        1           1      1        1   

        Nigeria  Pakistan  Saudi  Turkey  
Clause                                    
P1            1         1      1       1  
P2            1         1      1       1  
P3            1         1      1       1  
P4            1         1      1       1  
P5            1         1      1       1   (51, 11

com.brave.merge_1513
com.businesscardmanager.documentscanner_56
com.DefaultCompany.BloodBox_136
com.Flashlightonclap_76
com.haktansoft.giftcenter_83
com.HD.Wallpapers.Best.Wall.Backgrounds_691
com.headfone.www.headfone_70329
com.hecorat.screenrecorder.free_70329
com.herocraft.game.cut.arcade.catchthecandy_13202
com.hg.cartcrash_28
com.higame.par.diy.firework_21
com.highcore.hoopstars_144
com.brave.merge_1513
